## NLP Task - Build and evaluate a sentiment classifier

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 44.6 MB/s eta 0:00:00


In [3]:
import nltk
from nltk.corpus import movie_reviews
import gensim

nltk.download('movie_reviews')
docs = [(list(movie_reviews.words(fileid)),category)
for category in movie_reviews.categories()
for fileid in movie_reviews.fileids(category)]

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


In [4]:
import random
random.shuffle(docs)

In [5]:
def extract_features(words):
  return {word: True for word in words}

In [6]:
features_sets = [(extract_features(docs),category)for (docs, category)in docs]
train_set, test_set = features_sets[:1500], features_sets[1500:]

In [7]:
classifier = nltk.NaiveBayesClassifier.train(train_set)
print(nltk.classify.accuracy(classifier, test_set))

0.756


In [8]:
classifier.show_most_informative_features(10)

Most Informative Features
               marvelous = True              pos : neg    =     12.3 : 1.0
               ludicrous = True              neg : pos    =     12.3 : 1.0
                  seagal = True              neg : pos    =     11.1 : 1.0
                   sucks = True              neg : pos    =     11.1 : 1.0
              astounding = True              pos : neg    =     10.3 : 1.0
                  avoids = True              pos : neg    =     10.3 : 1.0
                  darker = True              pos : neg    =     10.3 : 1.0
                dazzling = True              pos : neg    =     10.3 : 1.0
                  regard = True              pos : neg    =     10.3 : 1.0
               animators = True              pos : neg    =      9.6 : 1.0


## NLP Task 2 - N-Gram phrase extraction with frequency filtering

In [11]:
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

words = movie_reviews.words()
finder = BigramCollocationFinder.from_words(words)

finder.apply_freq_filter(20)

print(finder.nbest(BigramAssocMeasures.pmi, 3))


[('del', 'toro'), ('salma', 'hayek'), ('san', 'francisco')]


### Word2vec based similarity search using NLTK corpus

In [12]:
nltk.download('brown')

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

In [15]:
import gensim
from nltk.corpus import brown

sentences = brown.sents()
model = gensim.models.Word2Vec(sentences,  vector_size= 100, window = 5, min_count = 1, sg = 1)

print(model.wv.most_similar('money'))

[('comfort', 0.8951388597488403), ('care', 0.8906254172325134), ('talent', 0.8876204490661621), ('joy', 0.886736273765564), ('satisfaction', 0.8843560814857483), ('friendship', 0.8843406438827515), ('job', 0.8837881088256836), ('someone', 0.882643461227417), ('threatening', 0.8815501928329468), ('nurse', 0.8804158568382263)]


In [16]:
### Doc2Vec for document classification

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import accuracy_score

docs = [TaggedDocument(words=movie_reviews.words(fileid), tags=[fileid])
for  fileid in (movie_reviews.fileids())]

model=Doc2Vec(docs, vector_size=50, epochs=30)

x=[model.dv[doc.tags[0]] for doc in docs]
y=[movie_reviews.categories(fileid)[0] for fileid in movie_reviews.fileids()]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

clf = LogisticRegression()
clf.fit(x_train, y_train)

print(accuracy_score(y_test, clf.predict(x_test)))

0.72


### NER with Spacy

In [17]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [20]:
from spacy import displacy

doc = nlp("this is aggreement is made on 4th April 2025 between Apple and GOI")
displacy.render(doc, style="ent", jupyter = True)

In [22]:
from sklearn.cluster import KMeans
import numpy as np
sentences=[
    "The cat sits on the mat",
    "Dogs bark loudly",
    "The dog is in  the yard"
]
docs = [nlp(sent) for sent in sentences]

x=np.array([doc.vector for doc in docs])

kmeans = KMeans(n_clusters=2)
kmeans.fit(x)

for i, label in enumerate(kmeans.labels_):
  print(f"cluster{label}: {sentences[i]}")

cluster0: The cat sits on the mat
cluster1: Dogs bark loudly
cluster0: The dog is in  the yard
